In [8]:
import os
import pandas as pd
import numpy as np
import random
import dendropy
import treeswift
from treeswift import read_tree_newick
#import matplotlib.pyplot as plt
import re

In [9]:
tree_path = "/Users/nora/Documents/ml_metagenomics/tol_variable_k_resuls/revisions_within_across_dist_longest_contigs/k7_v39_8k_s28_clade_All_TOL_Contigs_Global_logest"

#grep -v "^[# ]" combined_distances.newick | grep -v "^$" > cleaned_tree.nwk
#sed -E 's/\[.*\]//g' cleaned_tree.nwk > cleaned_no_meta.nwk

tree_file = "combined_distances.newick"

tree = treeswift.read_tree_newick(os.path.join(tree_path, tree_file))

# Build a set of leaf labels containing ".part"
labels_to_keep = {leaf.label for leaf in tree.traverse_leaves() if ".part" in leaf.label}

tree2 = tree.extract_tree_with(labels_to_keep, suppress_unifurcations=False)

In [10]:
#print("Number of leaves after pruning:", tree2.num_leaves())

In [11]:
M = tree2.distance_matrix(leaf_labels=True)

In [12]:
len(list(M.keys()))

2262

In [13]:
len(M["G000214595.part_NZ_AFIG01000003.1"])

2261

In [14]:
# Build pandas DataFrame
leaves = list(M.keys())
df = pd.DataFrame(index=leaves, columns=leaves, dtype=float)

for i in leaves:
    for j in leaves:
        if i == j:
            df.loc[i, j] = 0.0   # distance to self
        else:
            df.loc[i, j] = M[i][j]

# Optional: save to CSV/TSV
df.to_csv(os.path.join(tree_path, "pairwise_distances_kf2vec_after_placement.tsv"), sep="\t")

In [15]:
df.head(n=5)

,G001918745.part_MNGN01000045.1,G001918745.part_MNGN01000031.1,G000722915.part_JOKN01000001.1,G000724145.part_JNVL01000005.1,G001717035.part_MAGT01000010.1,G001717035.part_MAGT01000011.1,G001717035.part_MAGT01000007.1,G001717035.part_MAGT01000008.1,G001717035.part_MAGT01000009.1,G001717035.part_MAGT01000006.1,...,G000384955.part_ARRB01000001.1,G000379225.part_KB900870.1,G000714715.part_NZ_JMFO01000004.1,G000714715.part_NZ_JMFO01000002.1,G001263205.part_CP012333.1,G000487995.part_NZ_KI530591.1,G000487995.part_NZ_KI530590.1,G000714715.part_NZ_JMFO01000008.1,G000487995.part_NZ_KI530573.1,G000224765.part_AGAU01000083.1
G001918745.part_MNGN01000045.1,0.000000,25.918951,269.863279,276.597647,227.152244,227.894332,236.506606,237.773972,262.596570,231.445886,...,362.005036,369.703077,273.314689,286.747141,191.606542,256.876976,261.201146,280.200952,280.438641,247.544815
G001918745.part_MNGN01000031.1,25.918951,0.000000,295.782230,302.516598,253.071195,253.813283,262.425557,263.692923,288.515521,257.364837,...,387.923987,395.622028,299.233640,312.666092,217.525493,282.795927,287.120097,306.119903,306.357592,273.463766
G000722915.part_JOKN01000001.1,269.863279,295.782230,0.000000,92.692196,270.891257,271.633345,280.245619,281.512985,306.335583,275.184899,...,405.744049,413.442090,317.053702,330.486154,235.345555,300.615989,304.940159,323.939965,324.177654,291.283828
G000724145.part_JNVL01000005.1,276.597647,302.516598,92.692196,0.000000,277.625625,278.367713,286.979987,288.247353,313.069951,281.919267,...,412.478417,420.176458,323.788070,337.220522,242.079923,307.350357,311.674527,330.674333,330.912022,298.018196
G001717035.part_MAGT01000010.1,227.152244,253.071195,270.891257,277.625625,0.000000,0.742088,9.354362,10.621728,35.444326,51.452200,...,352.865310,360.563351,264.174963,277.607415,182.466816,247.737250,252.061420,271.061226,271.298915,238.405089


In [16]:
import pandas as pd
from itertools import combinations, product

# df is your pairwise distance matrix DataFrame
# Leaves labeled like "sample1.part_001", "sample2.part_010", etc.

# --- Step 1: group contigs by sample ID ---
sample_to_contigs = {}
for leaf in df.index:
    sample_id = leaf.split(".part")[0]  # extract sample ID
    sample_to_contigs.setdefault(sample_id, []).append(leaf)

# --- Step 2: compute within-sample distances ---
within_distances = []
records = []
for sample, contigs in sample_to_contigs.items():
    if len(contigs) < 2:
        continue  # no pairs to compute
    # all pairwise combinations of contigs in the same sample
    for a, b in combinations(contigs, 2):
        within_distances.append(df.loc[a, b])
        records.append({
            'Contig1': a,
            'Contig2': b,
            'Distance': df.loc[a, b],
            'Type': 'Within',
            'Sample1': sample,
            'Sample2': sample
        })

mean_within = sum(within_distances) / len(within_distances)
print("Mean within-sample distance:", mean_within)

# --- Step 3: compute across-sample distances ---
across_distances = []
samples = list(sample_to_contigs.keys())
for i in range(len(samples)):
    for j in range(i + 1, len(samples)):
        contigs_i = sample_to_contigs[samples[i]]
        contigs_j = sample_to_contigs[samples[j]]
        # all pairs between two different samples
        for a, b in product(contigs_i, contigs_j):
            across_distances.append(df.loc[a, b])
            records.append({
                'Contig1': a,
                'Contig2': b,
                'Distance': df.loc[a, b],
                'Type': 'Across',
                'Sample1': samples[i],
                'Sample2': samples[j]
            })

mean_across = sum(across_distances) / len(across_distances)
print("Mean across-sample distance:", mean_across)

# --- Step 4: save to text file ---
output_df = pd.DataFrame(records)

all_distances = output_df['Distance'].values
min_val = all_distances.min()
max_val = all_distances.max()
output_df['Distance_scaled'] = (output_df['Distance'] - min_val) / (max_val - min_val)
output_df['Condition'] ='kf2vec_after_placement'

output_df.to_csv(os.path.join(tree_path, "distances_kf2vec_placement_R.txt"), sep='\t', index=False)
print("Saved distances distances_kf2vec_placement.txt")


Mean within-sample distance: 58.155188578064525
Mean across-sample distance: 331.337442577567
Saved distances distances_kf2vec_placement.txt


In [36]:
mean_across/mean_within

3.2316239322737905

In [37]:
# To scale distances relative to the combined range
import numpy as np

# Convert to numpy arrays if not already
within = np.array(within_distances)
across = np.array(across_distances)

# Combine to get the global min and max
all_distances = np.concatenate([within, across])
min_val = all_distances.min()
max_val = all_distances.max()

# Scale each set relative to the combined range
within_scaled = (within - min_val) / (max_val - min_val)
across_scaled = (across - min_val) / (max_val - min_val)

# Example: check min/max after scaling
print("Within scaled:", within_scaled.min(), "-", within_scaled.max())
print("Across scaled:", across_scaled.min(), "-", across_scaled.max())


Within scaled: 0.0 - 0.8217103006440996
Across scaled: 0.0 - 1.0


In [38]:
mean_scaled_within = sum(within_scaled) / len(within_scaled)
print("Mean within-sample distance:", mean_scaled_within)
mean_scaled_across = sum(across_scaled) / len(across_scaled)
print("Mean across-sample distance:", mean_scaled_across)

Mean within-sample distance: 0.03433655863462039
Mean across-sample distance: 0.1109628446347678


In [39]:
mean_scaled_across/mean_scaled_within

3.231623932250675

3.2316239322737905

In [28]:
from scipy.stats import f_oneway

# within_distances and across_distances are your lists of pairwise distances
F_stat, p_value = f_oneway(within_distances, across_distances)

print("F-statistic:", F_stat)
print("p-value:", p_value)

F-statistic: 7820.833961820503
p-value: 0.0


In [11]:
# Save to CSV/TSV
#matrix.to_csv("pairwise_distances.tsv", sep="\t")

#print("Distance matrix saved. Shape:", matrix.shape)